In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    explode,
    to_timestamp,
)


BRONZE_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/bronze/rest/company_profiles/"
)

REFERENCE_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/reference/company_profiles/"
)


def main():

    spark = (
        SparkSession.builder
        .appName("CompanyProfilesWriter")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.shuffle.partitions", "4")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    # -----------------------------------------------------
    # Read Bronze JSON
    # -----------------------------------------------------

    bronze_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .json(BRONZE_PATH)
    )

    # Each Bronze file contains an array of company records
    exploded_df = (
        bronze_df
        .withColumn(
            "profile",
            explode(col("records"))
        )
    )

    # -----------------------------------------------------
    # Clean reference table
    # -----------------------------------------------------

    profiles_df = (
        exploded_df

        .select(
            col("profile.symbol")
                .alias("symbol"),

            col("profile.company_name")
                .alias("company_name"),

            col("profile.industry")
                .alias("industry"),

            col("profile.exchange")
                .alias("exchange"),

            col("profile.country")
                .alias("country"),

            col("profile.currency")
                .alias("currency"),

            col("profile.ipo_date")
                .alias("ipo_date"),

            col("profile.market_capitalization")
                .cast("double")
                .alias("market_capitalization"),

            col("profile.shares_outstanding")
                .cast("double")
                .alias("shares_outstanding"),

            col("profile.website")
                .alias("website"),

            col("profile.phone")
                .alias("phone"),

            col("profile.logo")
                .alias("logo"),

            to_timestamp(
                col("collected_at_utc")
            ).alias("collected_at"),

            col("source"),
            col("schema_version"),
        )

        # If we collected the same company more than once,
        # keep one reference record per symbol.
        .dropDuplicates(["symbol"])
    )

    print(
        f"Company profile records: "
        f"{profiles_df.count()}"
    )

    print("\nCompany profile schema:")
    profiles_df.printSchema()

    print("\nCompany profiles:")
    profiles_df.orderBy("symbol").show(
        truncate=False
    )

    # -----------------------------------------------------
    # Write Parquet
    # -----------------------------------------------------

    (
        profiles_df.write
        .mode("overwrite")
        .option("compression", "snappy")
        .parquet(REFERENCE_PATH)
    )

    print("=" * 60)
    print("Company profile reference write completed")
    print(f"Reference path: {REFERENCE_PATH}")
    print("=" * 60)

    spark.stop()


if __name__ == "__main__":
    main()